In [13]:
import joblib
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report

In [2]:
loaded_model = joblib.load("models/gradient_boosting_model.pk1")

print(loaded_model)

HistGradientBoostingClassifier(class_weight='balanced', random_state=42)


In [3]:
loaded_scaler = joblib.load("models/scaler.pk1")
print(loaded_scaler)

StandardScaler()


In [4]:
loaded_threshold = joblib.load("models/threshold.pk1")
print("Loaded threshold:", loaded_threshold)

Loaded threshold: 0.5500000000000002


Test the loaded model

In [7]:
df = pd.read_csv("data/final_trained_model.csv")

In [8]:
X = df.drop(columns=["target"])
y = df["target"]

In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(X_test.shape)

(261113, 180)


In [10]:
continuous_features = [
    col for col in X_train.columns
    if X_train[col].nunique() > 2
]

In [11]:
X_train[continuous_features] = loaded_scaler.transform(
    X_train[continuous_features]
)

X_test[continuous_features] = loaded_scaler.transform(
    X_test[continuous_features]
)

print("Scaling completed.")

Scaling completed.


In [12]:
y_prob = loaded_model.predict_proba(X_test)[:, 1]

final_pred = (y_prob >= loaded_threshold).astype(int)

print("Predictions completed.")

Predictions completed.


In [14]:
print(confusion_matrix(y_test, final_pred))
print(classification_report(y_test, final_pred))

[[153044  55483]
 [ 20819  31767]]
              precision    recall  f1-score   support

           0       0.88      0.73      0.80    208527
           1       0.36      0.60      0.45     52586

    accuracy                           0.71    261113
   macro avg       0.62      0.67      0.63    261113
weighted avg       0.78      0.71      0.73    261113



In [15]:
feature_columns = X_train.columns.tolist()
joblib.dump(feature_columns, "models/feature_columns.pkl")

joblib.dump(continuous_features, "models/continuous_features.pkl")

print("Preprocessing information saved successfully.")

Preprocessing information saved successfully.


Creating the prediction function

In [24]:
def predict_loan_risk(loan_data):
    loan = pd.DataFrame([loan_data])

    loan = loan.reindex(columns=feature_columns)
    loan[continuous_features] = loaded_scaler.transform(loan[continuous_features])
    probability = loaded_model.predict_proba(loan)[0,1]
    prediction = int(probability >= loaded_threshold)

    if prediction == 1:
        result = "Risky Loan"
    else:
        result = "Low-risk Loan"

    return probability, prediction, result

Creating one test loan

In [26]:
test_loan = X_test.iloc[0].to_dict()

probability, prediction, result = predict_loan_risk(test_loan)

print("Risk probability:", round(probability * 100, 2), "%")
print("Prediction:", prediction)
print("Result:", result)

Risk probability: 20.15 %
Prediction: 0
Result: Low-risk Loan


Compare prediction with the actual result

In [27]:
actual = y_test.iloc[0]

print("Actual:", actual)
print("Predicted:", prediction)

if actual == prediction:
    print("Result: Correct prediction")
else:
    print("Result: Incorrect prediction")

Actual: 0
Predicted: 0
Result: Correct prediction


Test it with several loans

In [29]:
for i in range(5):
    test_loan = X_test.iloc[i].to_dict()

    probability, prediction, result = predict_loan_risk(test_loan)
    actual = y_test.iloc[i]

    print(f"Loan {i + 1}")
    print("Risk probability:", round(probability * 100, 2), "%")
    print("Predicted:", prediction)
    print("Actual:", actual)
    print("Result:", result)
    print("-" * 40)

Loan 1
Risk probability: 20.15 %
Predicted: 0
Actual: 0
Result: Low-risk Loan
----------------------------------------
Loan 2
Risk probability: 29.59 %
Predicted: 0
Actual: 1
Result: Low-risk Loan
----------------------------------------
Loan 3
Risk probability: 25.76 %
Predicted: 0
Actual: 0
Result: Low-risk Loan
----------------------------------------
Loan 4
Risk probability: 26.23 %
Predicted: 0
Actual: 0
Result: Low-risk Loan
----------------------------------------
Loan 5
Risk probability: 26.24 %
Predicted: 0
Actual: 0
Result: Low-risk Loan
----------------------------------------
